# Wan2.2-TI2V-5B · Colab 文生视频

> ⚠️ **未在 GPU 上实测，首跑需在 Colab 里验证。** 本笔记本按 diffusers 官方用法编写（`WanPipeline`，模型 `Wan-AI/Wan2.2-TI2V-5B`，Apache 2.0）；参考的 Kaggle 笔记本用的是 Wan2GP 界面方案（T4 x2 跑通），Kaggle 页面只暴露了运行元信息，未读到源码，故此处改用更易复核的 diffusers 路径。
>
> - 运行时：Colab 免费 GPU **T4 16GB**。
# Qwen 首跑实测：T4 16GB 下整块搬运会 OOM，先改逐层卸载（2026-11-09，未跑待验证）
> - ⚠️ **显存警告**：官方参考称 TI2V-5B 单卡推理至少需要 24GB 显存，T4 16GB 低于官方下限。本笔记本用 `enable_sequential_cpu_offload()` + `enable_vae_tiling()` 压低峰值显存，480p / 33 帧是保守起点，仍可能很慢或 OOM——首跑即验证点。
> - 帧数必须是 `4n+1`（VAE 时间压缩 4×）：33 帧 ≈ 16fps 下约 2 秒；宽高须为 32 的倍数（TI2V-5B 要求）。
> - 结果保存：Google Drive `/MyDrive/free-stack/wan2.2-5b/`。

In [ ]:
# ===== 可改参数：只改这里 =====
PROMPT = "A golden retriever puppy running through shallow ocean waves at sunset, slow motion, cinematic"
NEGATIVE_PROMPT = "色调艳丽，过曝，静态，细节模糊，字幕，风格，作品，画作，画面，丑陋，错误的，最差的质量，低质量，JPEG伪影，PNSR，Watermark"
WIDTH = 832           # 480p，须为 32 的倍数
HEIGHT = 480
NUM_FRAMES = 33       # 须为 4n+1；33 帧 ≈ 2 秒（16fps）
STEPS = 30            # 30–50，越高越慢
GUIDANCE = 5.0        # 官方示例常用 5.0
SEED = 42
FPS = 16
OUT_DIR = "/content/drive/MyDrive/free-stack/wan2.2-5b"

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("输出目录:", OUT_DIR)

In [ ]:
!pip install -q git+https://github.com/huggingface/diffusers transformers accelerate imageio imageio-ffmpeg

In [ ]:
import torch
from diffusers import WanPipeline

pipe = WanPipeline.from_pretrained(
    "Wan-AI/Wan2.2-TI2V-5B",
    torch_dtype=torch.bfloat16,
)
# T4 16GB 低于官方 24GB 下限：CPU 卸载 + VAE 分块解码
# Qwen 首跑实测：T4 16GB 下整块搬运会 OOM，先改逐层卸载（2026-11-09，未跑待验证）
pipe.enable_sequential_cpu_offload()
pipe.enable_vae_tiling()
print("模型已加载（CPU offload + VAE tiling 模式）")

In [ ]:
import os, time
from datetime import datetime
from diffusers.utils import export_to_video

t0 = time.time()
video = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    height=HEIGHT,
    width=WIDTH,
    num_frames=NUM_FRAMES,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE,
    generator=torch.Generator("cpu").manual_seed(SEED),
).frames[0]
print("生成用时: %.1fs" % (time.time() - t0))

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
mp4_path = os.path.join(OUT_DIR, f"wan2.2-ti2v-5b_{stamp}_s{SEED}.mp4")
export_to_video(video, mp4_path, fps=FPS)
with open(os.path.join(OUT_DIR, f"wan2.2-ti2v-5b_{stamp}_s{SEED}.txt"), "w", encoding="utf-8") as f:
    f.write(f"prompt: {PROMPT}\nnegative_prompt: {NEGATIVE_PROMPT}\nsize: {WIDTH}x{HEIGHT}\nframes: {NUM_FRAMES}\nsteps: {STEPS}\nguidance: {GUIDANCE}\nseed: {SEED}\nfps: {FPS}\nmodel: Wan-AI/Wan2.2-TI2V-5B\n")
print("已保存:", mp4_path)

## 首跑检查清单

- 若 OOM：先把 `NUM_FRAMES` 降到 17（约 1 秒），或把分辨率降到 640×384；仍不行就说明 T4 16GB 确实带不动，改等 Colab 分到更大显存或换方案。
- 生成很慢是正常的（CPU 卸载 + VAE 解码），一支 2 秒 480p 短片在 T4 上可能要数分钟到十几分钟。
- 生成后去 Drive `free-stack/wan2.2-5b/` 核对 MP4 与同名 txt（prompt 记录）。
- 模型 Apache 2.0，生成内容可商用（以实际使用场景为准）。